# 07 — Train Llama-3.2-3B: QA (Nepali)

- **Model:** `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`
- **Dataset:** xquad.ne (~952 train examples)
- **Method:** QLoRA r=16 via Unsloth
- **Output:** `iwasbinod/llama32-3b-nepali-qa-qlora`

In [ ]:
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git -q
!pip install "trl>=0.12.0,<0.16.0" peft accelerate bitsandbytes -q

In [ ]:
import sys
sys.path.append('/kaggle/working/final_year_proj_finetuning')

from src.data_utils import load_jsonl
from src.training_utils import load_model, apply_qlora, build_trainer, save_and_push
from src.evaluation import evaluate_qa
from src.utils import set_seed, hf_login, print_gpu_memory
import os

set_seed(42)
print_gpu_memory()
hf_login(os.environ.get('HF_TOKEN'))

In [ ]:
train_data = load_jsonl('data/processed/qa_train.jsonl')
val_data   = load_jsonl('data/processed/qa_val.jsonl')
print(f'Train: {len(train_data)}, Val: {len(val_data)}')
print('\nSample prompt:')
print(train_data[0]['text'][:400])

In [ ]:
model, tokenizer = load_model('llama32-3b')
model = apply_qlora(model, r=16, lora_alpha=32)
print_gpu_memory()

In [ ]:
trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    output_dir='outputs/adapters/llama32-3b-nepali-qa-qlora',
    num_epochs=5,
    batch_size=2,
    grad_accum=4,
    lr=2e-4,
    max_seq_length=512,
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Done. Training loss: {trainer_stats.training_loss:.4f}')

In [ ]:
results = evaluate_qa(model, tokenizer, val_data, n_samples=100)
print(f'EM={results["exact_match"]}, F1={results["f1"]}')
print('\nSample predictions:')
for i in range(3):
    print(f'  Ref:  {results["references"][i]}')
    print(f'  Pred: {results["hypotheses"][i]}')
    print()

In [ ]:
repo_id = save_and_push(model, tokenizer, model_key='llama32-3b', task='qa')
print(f'Adapter at: https://huggingface.co/{repo_id}')